# Recalibration of the household WASH deprivation model

Put this notebook beside **Nigeria_DHS_2013_2018_2024_Households_Climate.csv.gz**, or leave the file inside **climate_stack_outputs**. Open the notebook in JupyterLab and choose **Run → Run All Cells**. Keep the input compressed. If a package is missing, run **%pip install scikit-learn scipy pandas numpy matplotlib pillow python-docx** in a code cell, restart the kernel, and run all cells again.

Results go to **wash_recalibration_outputs**. For automatic Word 97–2003 DOC export, install LibreOffice Writer. If unavailable, open the saved DOCX in desktop Word and choose **Save As → Word 97–2003 Document (*.doc)**.

The reference model uses household predictors only. The original 2018-trained predictions are kept as the comparator. The two adaptations use labelled 2024 households outside the test zone. These estimates evaluate adaptation with 2024 data. They apply to sampled households with a known deprivation outcome.


In [ ]:
from pathlib import Path
import os
import shutil
import subprocess
from tempfile import TemporaryDirectory

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from PIL import Image
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import average_precision_score, brier_score_loss, log_loss, roc_auc_score
from sklearn.preprocessing import StandardScaler
from scipy.optimize import brentq, minimize
from scipy.special import expit, logit

ROOT = Path.cwd()
INPUT = ROOT / "Nigeria_DHS_2013_2018_2024_Households_Climate.csv.gz"
if not INPUT.is_file():
    INPUT = ROOT / "climate_stack_outputs" / INPUT.name
if not INPUT.is_file():
    raise FileNotFoundError("Put the compressed climate-linked household file beside the notebook.")
OUTPUT = ROOT / "wash_recalibration_outputs"
OUTPUT.mkdir(exist_ok=True)
KEY = ["survey_round","hv001","hv002"]
TARGET = "wash_dep_2plus"
DEMOGRAPHIC = ["rural","head_female","head_age_years_capped","usual_household_size","nonwash_asset_count"]
RAW = ["head_age_years_capped","usual_household_size","nonwash_asset_count"]
columns = [*KEY,"state","zone","residence","head_sex","hh_weight","psu_key","stratum_hv023_key","wash_domains_known",TARGET,*RAW]
d = pd.read_csv(INPUT,usecols=columns,low_memory=False)
if d.groupby("survey_round").size().to_dict() != {2013:38522,2018:40427,2024:40047}:
    raise ValueError("Unexpected survey counts. Select the combined household output from the stack notebook.")
if d[KEY].isna().any().any() or d.duplicated(KEY).any():
    raise ValueError("Missing or duplicate household keys.")
if not np.isfinite(d.hh_weight).all() or not d.hh_weight.gt(0).all():
    raise ValueError("Household weights must be positive and finite.")
if not d[TARGET].dropna().isin([0,1]).all() or d.loc[d.survey_round.eq(2013),TARGET].notna().any():
    raise ValueError("The file does not use the verified main 2018–2024 target.")
main_rounds = d.survey_round.isin([2018,2024])
if not d.loc[main_rounds,TARGET].notna().eq(d.loc[main_rounds,"wash_domains_known"].eq(3)).all():
    raise ValueError("The target does not match the complete-case eligibility rule.")
if d[["state","zone"]].isna().any().any() or d.groupby("state").zone.nunique().ne(1).any():
    raise ValueError("Each state needs a consistent zone label.")
for column in RAW:
    d[column] = pd.to_numeric(d[column],errors="raise")
    if not np.isfinite(d[column].dropna()).all():
        raise ValueError(f"Non-finite predictor values in {column}.")
if not d.residence.dropna().isin(["Rural","Urban"]).all() or not d.head_sex.dropna().isin(["Female","Male"]).all():
    raise ValueError("Unexpected residence or head-sex labels.")
d["rural"] = d.residence.map({"Rural":1.,"Urban":0.})
d["head_female"] = d.head_sex.map({"Female":1.,"Male":0.})
analysis = d.loc[main_rounds & d[TARGET].notna()].copy()
analysis[TARGET] = analysis[TARGET].astype(int)
if analysis.groupby("survey_round").size().to_dict() != {2018:36821,2024:36412}:
    raise ValueError("Unexpected complete-case totals. Review the target and household input.")
audit = []
for year,g in d.groupby("survey_round"):
    known = g[TARGET].notna()
    audit.append({
        "survey_round":year, "households":len(g), "known_target":int(known.sum()),
        "missing_target":int((~known).sum()), "weighted_target_known_pct":100*g.loc[known,"hh_weight"].sum()/g.hh_weight.sum(),
    })
audit = pd.DataFrame(audit)
audit.to_csv(OUTPUT/"Household_Model_Input_QC.csv",index=False)
display(audit)


## 1. Fit the reference model and the calibration adjustments

For each test zone, fit the household-only logistic model on 2018 data from the other five zones. Freeze that model, then use its scores on the other five zones' 2024 households to fit the calibration adjustments. Apply both adjustments to the held-out 2024 zone. The held-out zone supplies neither training households nor calibration outcomes.

The intercept-only adjustment shifts probabilities up or down. The intercept-and-slope adjustment also changes how extreme the probabilities are. Both are fit with the 2024 household weights. The unadjusted comparator uses the same 2018 model and test households.


In [ ]:
ZONES = ["North Central","North East","North West","South East","South South","South West"]
METHODS = ["Original","Intercept only","Intercept and slope"]
train2018 = analysis.loc[analysis.survey_round.eq(2018)]
test2024 = analysis.loc[analysis.survey_round.eq(2024)].copy()
if set(train2018.zone) != set(ZONES) or set(test2024.zone) != set(ZONES):
    raise ValueError("The six zones must appear in both modelling rounds.")

def fit_base(train):
    median = train[DEMOGRAPHIC].median()
    if median.isna().any():
        raise ValueError("A training predictor is entirely missing.")
    x = train[DEMOGRAPHIC].fillna(median)
    scaler = StandardScaler().fit(x)
    model = LogisticRegression(C=1,max_iter=1000)
    model.fit(scaler.transform(x),train[TARGET],sample_weight=train.hh_weight/train.hh_weight.mean())
    return median,scaler,model

def score_base(frame, fitted):
    median,scaler,model = fitted
    x = scaler.transform(frame[DEMOGRAPHIC].fillna(median))
    return model.predict_proba(x)[:,1]

def calibration_fit(z,y,w):
    if np.unique(y).size != 2 or np.ptp(z)==0:
        raise ValueError("Calibration needs both outcomes and varying base predictions.")
    w = w / w.sum()
    a = brentq(lambda a: np.dot(w,expit(z+a)-y),-60,60)
    x = np.column_stack([np.ones(len(z)),z])
    def loss(beta):
        eta = x @ beta
        return np.dot(w,np.logaddexp(0,eta)-y*eta), x.T @ (w*(expit(eta)-y))
    result = minimize(loss,[a,1.],jac=True,method="BFGS",options={"gtol":1e-9})
    if not np.isfinite(result.x).all() or np.max(np.abs(result.jac))>1e-7:
        raise RuntimeError("Intercept-and-slope recalibration did not converge.")
    return a,*result.x

def summarize(test,predicted):
    y,w = test[TARGET],test.hh_weight
    if y.nunique() != 2:
        raise ValueError("The holdout must contain both outcome classes.")
    if not np.isfinite(predicted).all() or not ((predicted>=0)&(predicted<=1)).all():
        raise ValueError("Invalid predicted probabilities.")
    return {
        "n_test":len(test),"observed_prevalence":np.average(y,weights=w),
        "mean_prediction":np.average(predicted,weights=w),
        "auc":roc_auc_score(y,predicted,sample_weight=w),
        "average_precision":average_precision_score(y,predicted,sample_weight=w),
        "brier":brier_score_loss(y,predicted,sample_weight=w),
        "log_loss":log_loss(y,predicted,sample_weight=w,labels=[0,1]),
    }



predictions = pd.DataFrame(index=test2024.index,columns=METHODS,dtype=float)
fold_rows,zone_rows = [],[]
for zone in ZONES:
    train = train2018.loc[train2018.zone.ne(zone)]
    calibration = test2024.loc[test2024.zone.ne(zone)]
    test = test2024.loc[test2024.zone.eq(zone)]
    if (set(train.state) | set(calibration.state)) & set(test.state):
        raise ValueError(f"State leakage in the {zone} fold.")
    if set(calibration.index) & set(test.index) or predictions.loc[test.index].notna().any().any():
        raise ValueError(f"Overlapping calibration or test cases for {zone}.")
    fitted = fit_base(train)
    raw_cal = score_base(calibration,fitted)
    raw_test = score_base(test,fitted)
    zcal = logit(np.clip(raw_cal,1e-8,1-1e-8))
    ztest = logit(np.clip(raw_test,1e-8,1-1e-8))
    a,intercept,slope = calibration_fit(zcal,calibration[TARGET].to_numpy(),calibration.hh_weight.to_numpy())
    values = dict(zip(METHODS,[raw_test,expit(ztest+a),expit(intercept+slope*ztest)]))
    fold_rows.append({"test_zone":zone,"train_2018_n":len(train),"calibration_2024_n":len(calibration),
                      "test_2024_n":len(test),"intercept_only":a,"intercept_joint":intercept,"slope_joint":slope})
    for method,p in values.items():
        predictions.loc[test.index,method] = p
        row = summarize(test,p)
        zone_rows.append({"zone":zone,"method":method,**row,
                          "bias_pp":100*(row["mean_prediction"]-row["observed_prevalence"])})
if predictions.isna().any().any():
    raise ValueError("A 2024 test household lacks predictions.")
folds = pd.DataFrame(fold_rows)
zones = pd.DataFrame(zone_rows)
pooled = pd.DataFrame([{"method":method,**summarize(test2024,predictions[method].to_numpy())}
                       for method in METHODS])
pooled["bias_pp"] = 100*(pooled.mean_prediction-pooled.observed_prevalence)
display(pooled.round(4))


## 2. Compare Brier scores, bias and calibration

Lower Brier score and log loss indicate better probability predictions. Positive bias means overestimation. The paired Brier differences subtract the original model's score from the adjusted model's score. Each interval uses the same 999 rescaled 2024 PSU bootstrap replicates, sampled within survey strata. The intervals condition on the fitted models and calibration adjustments. They omit calibration and training uncertainty and missing-outcome bias.

The calibration curve groups held-out predictions into probability bins, and compares each bin's weighted mean prediction with its weighted observed prevalence. Results are evaluated on held-out zones after using outcomes from the other zones in 2024.

References include [calibration of prediction models](https://doi.org/10.1186/s12916-019-1466-7) and the [rescaled survey bootstrap](https://dennisfeehan.org/surveybootstrap/articles/rescaled_bootstrap.html).


In [ ]:
PSU, STRATUM = "psu_key","stratum_hv023_key"
full2024 = d.loc[d.survey_round.eq(2024),[PSU,STRATUM]]
if full2024.isna().any().any():
    raise ValueError("2024 design identifiers contain missing values.")
design = full2024.drop_duplicates().sort_values([STRATUM,PSU]).reset_index(drop=True)
if design[PSU].duplicated().any():
    raise ValueError("A 2024 PSU belongs to more than one stratum.")
if design.groupby(STRATUM).size().lt(2).any():
    raise ValueError("This bootstrap requires at least two PSUs per 2024 stratum.")

B = 999
rng = np.random.default_rng(20241112)
rep = np.zeros((B,len(design)))
for _,members in design.groupby(STRATUM,sort=False):
    positions = members.index.to_numpy()
    n = len(positions)
    rep[:,positions] = rng.multinomial(n-1,np.repeat(1/n,n),size=B)*n/(n-1)

psu_code = test2024[PSU].map(pd.Series(design.index,index=design[PSU]))
if psu_code.isna().any():
    raise ValueError("Could not find a test household's PSU.")
codes = psu_code.to_numpy(dtype=int)
w = test2024.hh_weight.to_numpy()
y = test2024[TARGET].to_numpy()

def totals(values, group_codes=codes):
    return np.bincount(group_codes,weights=values,minlength=len(design))

total_weights = totals(w)
rep_total = rep @ total_weights
if (rep_total<=0).any():
    raise ValueError("A replicate has no eligible test households.")


loss_reps = {}
calibration_rows = []
for method in METHODS:
    p = predictions[method].to_numpy()
    error = p-y
    loss_reps[method] = (rep @ totals(w*error**2))/rep_total
    bias = 100*(rep @ totals(w*error))/rep_total
    row = pooled.method.eq(method)
    pooled.loc[row,["brier_ci_low","brier_ci_high"]] = np.quantile(loss_reps[method],[.025,.975])
    pooled.loc[row,["bias_ci_low_pp","bias_ci_high_pp"]] = np.quantile(bias,[.025,.975])
    frame = test2024[[TARGET,"hh_weight"]].assign(predicted=p)
    frame["bin"] = pd.cut(frame.predicted,bins=np.linspace(0,1,11),include_lowest=True)
    for label,g in frame.groupby("bin",observed=True):
        calibration_rows.append({"method":method,"bin":str(label),"n":len(g),
                                 "predicted":np.average(g.predicted,weights=g.hh_weight),
                                 "observed":np.average(g[TARGET],weights=g.hh_weight)})

by_method = pooled.set_index("method")
paired = []
for method in METHODS[1:]:
    low,high = np.quantile(loss_reps[method]-loss_reps["Original"],[.025,.975])
    paired.append({"method":method,"brier_change":by_method.loc[method,"brier"]-by_method.loc["Original","brier"],
                   "ci95_low":low,"ci95_high":high,
                   "absolute_bias_reduction_pp":abs(by_method.loc["Original","bias_pp"])-abs(by_method.loc[method,"bias_pp"])})
paired = pd.DataFrame(paired)
curves = pd.DataFrame(calibration_rows)
assert len(zones)==18 and folds.test_2024_n.sum()==36412
assert np.allclose(rep.sum(axis=1),len(design))
for filename,table in {
    "Recalibration_Pooled_Metrics.csv":pooled,
    "Recalibration_Zone_Metrics.csv":zones,
    "Recalibration_Paired_Brier.csv":paired,
    "Recalibration_Fold_Parameters.csv":folds,
    "Recalibration_Curve_Bins.csv":curves,
}.items():
    table.to_csv(OUTPUT/filename,index=False)
display(paired.round(4))


## 3. Save the report and figures

Open **wash_recalibration_outputs** and start with the Word report. Check **Recalibration_Paired_Brier.csv** for lower Brier scores and **Recalibration_Zone_Metrics.csv** for zones where adjustment leaves substantial bias. These checks concern adaptation to the 2024 survey and do not validate a deployment for a future survey or for unsampled populations.


In [ ]:
dpi = 600

def save_figure(figure, filename):
    path = OUTPUT / filename
    figure.savefig(path, format="jpeg", dpi=dpi, facecolor="white",
                   pil_kwargs={"quality": 95, "subsampling": 0})
    plt.close(figure)
    with Image.open(path) as image:
        if image.format != "JPEG" or image.info.get("dpi") != (dpi, dpi):
            raise ValueError(f"Could not verify JPEG resolution for {path.name}.")
    return path




colors = ["#626870","#176C87","#A34D20"]
fig,ax = plt.subplots(figsize=(7,3.45),layout="constrained")
ax.plot([0,1],[0,1],"--",color="#A0A6AA",linewidth=1,label="Perfect calibration")
for method,color in zip(METHODS,colors):
    frame = curves.loc[curves.method.eq(method)].sort_values("predicted")
    ax.plot(frame.predicted,frame.observed,"o-",color=color,markersize=3,label=method)
ax.set(xlim=(0,1),ylim=(0,1),xlabel="Weighted mean prediction",ylabel="Weighted observed proportion",
       title="Calibration in held-out 2024 zones")
ax.grid(color="#E3E7EB")
ax.legend(frameon=False,fontsize=8)
cal_fig = save_figure(fig,"Figure_1_Held_Out_Calibration_600dpi.jpg")

fig,ax = plt.subplots(figsize=(7,3.35),layout="constrained")
for i,(method,color) in enumerate(zip(METHODS,colors)):
    frame = zones.loc[zones.method.eq(method)].set_index("zone").loc[ZONES]
    ax.scatter(np.arange(len(ZONES))+(i-1)*.15,frame.bias_pp,s=24,color=color,label=method)
ax.axhline(0,linestyle="--",color="#A0A6AA",linewidth=1)
ax.set(xticks=np.arange(len(ZONES)),xticklabels=ZONES,ylabel="Predicted minus observed (pp)",
       title="Prediction bias by held-out zone")
ax.tick_params(axis="x",labelsize=8)
ax.grid(axis="y",color="#E3E7EB")
ax.legend(frameon=False,fontsize=8,ncol=3)
bias_fig = save_figure(fig,"Figure_2_Recalibration_Zone_Bias_600dpi.jpg")
from docx import Document
from docx.enum.table import WD_TABLE_ALIGNMENT, WD_CELL_VERTICAL_ALIGNMENT
from docx.enum.text import WD_ALIGN_PARAGRAPH
from docx.oxml import OxmlElement
from docx.oxml.ns import qn
from docx.shared import Inches, Pt, RGBColor

figure_width = 7
doc = Document()
doc.core_properties.title = "Recalibration of household WASH predictions in Nigeria"
doc.core_properties.author = ""
section = doc.sections[0]
section.page_width, section.page_height = Inches(8.5), Inches(11)
section.left_margin = section.right_margin = Inches(0.75)
section.top_margin, section.bottom_margin = Inches(0.7), Inches(0.7)

for style_name, size in [("Normal", 10.5), ("Title", 25), ("Heading 1", 17), ("Heading 2", 13), ("Caption", 9)]:
    style = doc.styles[style_name]
    style.font.name = "Calibri"
    style.font.size = Pt(size)
    style.font.color.rgb = RGBColor.from_string("000000")
    style.paragraph_format.space_before = Pt(5)
    style.paragraph_format.space_after = Pt(5)
    if style_name == "Title":
        border = style.element.get_or_add_pPr().find(qn("w:pBdr"))
        if border is not None:
            border.getparent().remove(border)

def xml(parent, tag, **attributes):
    element = OxmlElement(f"w:{tag}")
    for key, value in attributes.items():
        element.set(qn(f"w:{key}"), str(value))
    parent.append(element)
    return element

def add_table(caption, headers, rows, widths, text_columns, font_size=9.5):
    doc.add_paragraph(caption, style="Caption").paragraph_format.keep_with_next = True
    table = doc.add_table(rows=0, cols=len(headers))
    table.alignment = WD_TABLE_ALIGNMENT.CENTER
    table.autofit = False
    for column, width in zip(table.columns, widths):
        column.width = Inches(width)
    props = table._tbl.tblPr
    borders = xml(props, "tblBorders")
    for edge in ("top", "left", "bottom", "right", "insideH", "insideV"):
        xml(borders, edge, val="single", sz=4, color="D9D9D9")
    margins = xml(props, "tblCellMar")
    for edge, value in (("top", 16), ("bottom", 16), ("left", 85), ("right", 85)):
        xml(margins, edge, w=value, type="dxa")

    for i, values in enumerate([headers, *rows]):
        row = table.add_row()
        row_props = row._tr.get_or_add_trPr()
        xml(row_props, "cantSplit")
        if i == 0:
            xml(row_props, "tblHeader")
        for j, (cell, value, width) in enumerate(zip(row.cells, values, widths)):
            cell.width = Inches(width)
            cell.vertical_alignment = WD_CELL_VERTICAL_ALIGNMENT.CENTER
            if i == 0 or i % 2 == 0:
                xml(cell._tc.get_or_add_tcPr(), "shd", fill="203A50" if i == 0 else "F2F6F9")
            paragraph = cell.paragraphs[0]
            paragraph.alignment = WD_ALIGN_PARAGRAPH.LEFT if j in text_columns else WD_ALIGN_PARAGRAPH.CENTER
            paragraph.paragraph_format.space_after = Pt(0)
            paragraph.paragraph_format.space_before = Pt(0)
            paragraph.paragraph_format.line_spacing = 1.05
            run = paragraph.add_run(str(value))
            run.bold = i == 0
            run.font.size = Pt(font_size)
            if i == 0:
                run.font.color.rgb = RGBColor.from_string("FFFFFF")
    return table

def add_figure(path, caption, description):
    paragraph = doc.add_paragraph()
    paragraph.paragraph_format.space_before = Pt(7)
    paragraph.paragraph_format.keep_with_next = True
    paragraph.alignment = WD_ALIGN_PARAGRAPH.CENTER
    shape = paragraph.add_run().add_picture(str(path), width=Inches(figure_width))
    shape._inline.docPr.set("descr", description)
    doc.add_paragraph(caption, style="Caption")



doc.add_paragraph("Recalibration of household WASH predictions in Nigeria",style="Title")
doc.add_paragraph("Nigeria Demographic and Health Survey 2018 and 2024")
doc.add_paragraph(
    "For each of six held-out zones, the reference model uses 2018 households from the other five zones. "
    "The two adjustments use labelled 2024 households from those five zones. The held-out zone supplies the "
    "test outcomes only. All three methods are evaluated on the same 36,412 eligible 2024 households."
)
doc.add_heading("Held out performance",level=1)
rows = [[r.method,f"{100*r.mean_prediction:.1f}",f"{r.bias_pp:+.1f}",f"{r.auc:.3f}",f"{r.brier:.4f}",f"{r.log_loss:.4f}"]
        for r in pooled.itertuples(index=False)]
add_table("Table 1. Weighted performance across the six held-out zones",
          ["Method","Mean prediction %","Bias (pp)","AUC","Brier","Log loss"],
          rows,[1.55,1.40,1.0,.85,1.05,1.15],{0})
rows = [[r.method,f"{r.brier_change:+.4f}",f"{r.ci95_low:+.4f} to {r.ci95_high:+.4f}",f"{r.absolute_bias_reduction_pp:.1f}"]
        for r in paired.itertuples(index=False)]
add_table("Table 2. Adjustment compared with the original model",
          ["Adjustment","Brier difference","95% interval","Absolute bias reduction (pp)"],
          rows,[1.50,1.40,1.95,2.15],{0})
doc.add_paragraph(
    f"Observed weighted prevalence was {100*pooled.observed_prevalence.iloc[0]:.1f}%. "
    "Negative Brier differences favour the adjustment. Positive bias means overestimation. "
    "Intervals reflect test-sample uncertainty with fitted models and adjustments held fixed."
)
doc.add_heading("Calibration and zone bias",level=1).paragraph_format.page_break_before=True
add_figure(cal_fig,"Figure 1. Weighted calibration among eligible households in held-out 2024 zones.",
           "Three reliability curves compared with the perfect-calibration diagonal.")
add_figure(bias_fig,"Figure 2. Weighted prediction bias in each held-out zone.",
           "Dot plot of signed bias for the original model and two calibration adjustments.")
def find_soffice():
    custom = os.environ.get("LIBREOFFICE_PATH")
    if custom and Path(custom).is_file():
        return custom
    candidates = [shutil.which("soffice"), shutil.which("libreoffice"),
                  "/Applications/LibreOffice.app/Contents/MacOS/soffice"]
    for variable in ("ProgramFiles", "ProgramFiles(x86)"):
        if base := os.environ.get(variable):
            candidates.append(str(Path(base) / "LibreOffice" / "program" / "soffice.com"))
    return next((candidate for candidate in candidates if candidate and Path(candidate).is_file()), None)


docx_path = OUTPUT / "Nigeria_DHS_WASH_Recalibration_Report.docx"
doc.save(docx_path)

def export_legacy_word(source, soffice):
    with TemporaryDirectory() as scratch:
        temp = Path(scratch).resolve()
        result = subprocess.run(
            [soffice, f"-env:UserInstallation={(temp / 'profile').as_uri()}", "--headless",
             "--convert-to", "doc:MS Word 97", "--outdir", str(temp), str(source.resolve())],
            capture_output=True, text=True, timeout=180,
        )
        converted = temp / source.with_suffix(".doc").name
        if result.returncode or not converted.is_file():
            raise RuntimeError(result.stderr or result.stdout or "Converter returned no file.")
        with converted.open("rb") as stream:
            if stream.read(8) != bytes.fromhex("D0CF11E0A1B11AE1"):
                raise ValueError("The converter did not create a binary Word 97–2003 document.")
        shutil.copyfile(converted, source.with_suffix(".doc"))

soffice = find_soffice()
if soffice:
    try:
        export_legacy_word(docx_path, soffice)
        print(f"Saved {docx_path.with_suffix('.doc').resolve()}")
    except (RuntimeError, ValueError, OSError, subprocess.TimeoutExpired) as error:
        print(f"Word .doc conversion failed: {error}")
        print("Open the saved .docx in desktop Word and save it as Word 97–2003 Document (*.doc).")
else:
    print("LibreOffice was unavailable. Open the saved .docx in desktop Word and save it as Word 97–2003 Document (*.doc).")


print(f"Saved the report, CSVs and two 600 dpi JPEGs in {OUTPUT.resolve()}.")
